#### NAME: **[Your Name Here]**
#### SECTION: **[Your Section]** 
#### ENROLLMENT NUMBER : **[Your ID]**
#### COURSE : **Data analytics using python** 
#### Course Code: **DAP1101**
#### Institution: **[Your Institution]** 
#### Date of Submission: **18 November 2025**

#### ABOUT PROJECT :
##### This mini project analyzes a dataset of Spotify tracks to identify key drivers of song popularity. The dataset includes audio features, artist metrics, and album details.
##### **Objective:** 
1. Analyze how artist fame (`artist_popularity`, `followers`) correlates with track success.
2. Determine if factors like `explicit` content or `release_year` influence popularity.
3. Build a **Regression Model** to predict the `track_popularity` score (0-100) based on these features.

## Step 1: Problem Definition & Dataset Selection

**Dataset Source:** Spotify Data (`spotify_data clean.csv`)

**Description:**
- **Type:** Structured CSV Data
- **Features:** 
 - **Numerical:** `track_duration_min`, `artist_popularity`, `artist_followers`, `album_total_tracks`.
 - **Categorical/Text:** `explicit`, `artist_genres`, `album_type`.
 - **Temporal:** `album_release_date`.
- **Target Variable:** `track_popularity` (Continuous, 0-100).

In [ ]:
# Import Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Import Scikit-Learn for Modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Set Visualization Style
sns.set(style="whitegrid")
plt.style.use('fivethirtyeight')

In [ ]:
# Load the Dataset
filename = 'spotify_data clean.csv'
try:
 # Attempt standard load
 df = pd.read_csv(filename)
 print("Dataset loaded successfully.")
 print(f"Initial Shape: {df.shape}")
except FileNotFoundError:
 print(f"Error: File '{filename}' not found. Please upload the dataset.")
except Exception as e:
 print(f"An error occurred: {e}")

# Display first 5 rows
df.head()

## Step 2: Data Cleaning & Preparation

**Preprocessing Logic:**
1. **Robust Explicit Handling:** The `explicit` column contains 'TRUE'/'FALSE' strings. We convert these to integers (1/0).
2. **Date Parsing:** We extract `release_year` from `album_release_date` to use as a numerical feature.
3. **Log Transformation:** `artist_followers` is heavily skewed (some artists have millions, most have few). We apply `np.log1p` to normalize this.
4. **Genre Extraction:** `artist_genres` is a string list. We create binary flags for major genres (`Pop`, `Rap`, `Rock`) instead of dropping the column.
5. **Missing Values:** We fill missing text fields with "Unknown" and drop rows only if critical numerical targets are missing.

In [ ]:
# 1. Handle 'Explicit' Column (String/Boolean to Integer)
def clean_explicit(val):
 # Convert to string, uppercase, and check
 s_val = str(val).upper().strip()
 if s_val in ['TRUE', '1', 'YES']:
 return 1
 return 0

df['explicit'] = df['explicit'].apply(clean_explicit)

# 2. Handle Dates (Extract Year)
# Convert to datetime, coerce errors to NaT
df['release_year'] = pd.to_datetime(df['album_release_date'], errors='coerce').dt.year

# Fallback: If standard parsing failed, try extracting first 4 digits from string
mask_missing_year = df['release_year'].isna()
df.loc[mask_missing_year, 'release_year'] = pd.to_numeric(
 df.loc[mask_missing_year, 'album_release_date'].astype(str).str[:4], errors='coerce'
)

# 3. Feature Engineering: Log Transform Followers
# Use log1p (log(1+x)) to handle 0s and reduce skew
df['log_artist_followers'] = np.log1p(df['artist_followers'])

# 4. Feature Engineering: Extract Simple Genres
# Fill NaN genres first
df['artist_genres'] = df['artist_genres'].fillna('Unknown')

# Create binary flags
df['is_pop'] = df['artist_genres'].astype(str).str.contains('pop', case=False).astype(int)
df['is_rap'] = df['artist_genres'].astype(str).str.contains('rap|hip hop', case=False).astype(int)
df['is_rock'] = df['artist_genres'].astype(str).str.contains('rock|metal', case=False).astype(int)

# 5. Encode Categorical: Album Type
le = LabelEncoder()
df['album_type_encoded'] = le.fit_transform(df['album_type'].astype(str))

# 6. Final Cleanup
# Drop identifiers and intermediate columns
cols_to_drop = ['track_id', 'track_name', 'artist_name', 'artist_genres', 
 'album_id', 'album_name', 'album_release_date', 'album_type', 'artist_followers']
df_model = df.drop(columns=cols_to_drop)

# Drop any remaining rows with NaNs in critical columns
df_model.dropna(inplace=True)

print(f"Data Cleaned. Final Shape for Modeling: {df_model.shape}")
df_model.head()

## Step 3: Exploratory Data Analysis (EDA)

We visualize distributions and correlations to understand the data before modeling.

In [ ]:
# 1. Univariate Analysis: Target Variable (Track Popularity)
plt.figure(figsize=(10, 5))
sns.histplot(df['track_popularity'], kde=True, color='#1DB954', bins=30)
plt.title('Distribution of Track Popularity Scores')
plt.xlabel('Popularity (0-100)')
plt.show()

In [ ]:
# 2. Bivariate Analysis: Artist Popularity vs Track Popularity
plt.figure(figsize=(10, 6))
sns.scatterplot(x='artist_popularity', y='track_popularity', data=df, alpha=0.6, color='purple')
plt.title('Correlation: Artist Fame vs. Song Success')
plt.xlabel('Artist Popularity Score')
plt.ylabel('Track Popularity Score')
plt.show()

In [ ]:
# 3. Multivariate Analysis: Correlation Matrix
plt.figure(figsize=(12, 10))
corr = df_model.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Features')
plt.show()

## Step 4: Statistical Inference

**Hypothesis Test (Independent T-Test):**
- **Null Hypothesis ($H_0$):** There is no difference in average popularity between **Explicit** and **Non-Explicit** songs.
- **Alternative Hypothesis ($H_1$):** There is a significant difference.

In [ ]:
# Define groups
group_explicit = df[df['explicit'] == 1]['track_popularity']
group_clean = df[df['explicit'] == 0]['track_popularity']

# Perform T-test
t_stat, p_val = stats.ttest_ind(group_explicit, group_clean, equal_var=False)

print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_val:.4f}")

# Interpretation
if p_val < 0.05:
 print("Result: Reject Null Hypothesis. Significant difference found.")
else:
 print("Result: Fail to reject Null Hypothesis. No significant difference.")

## Step 5: Modeling (Regression Analysis)

We use a **Random Forest Regressor** to predict popularity.
We also implement **Cross-Validation** to ensure the model is robust and not overfitting.

In [ ]:
# 1. Split Data
X = df_model.drop(['track_popularity'], axis=1)
y = df_model['track_popularity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Scale Features (Good practice for comparison)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# 4. Predict
y_pred = rf_model.predict(X_test_scaled)

print("Model Trained Successfully.")

In [ ]:
# 5. Evaluate Performance
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R-Squared ($R^2$): {r2:.4f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

# 6. Cross-Validation (Robustness Check)
cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='r2')
print(f"\nAverage CV R2 Score: {cv_scores.mean():.4f} (Std: {cv_scores.std():.4f})")

In [ ]:
# 7. Feature Importance Visualization
importances = pd.Series(rf_model.feature_importances*, index=X.columns)
plt.figure(figsize=(10, 6))
importances.nlargest(10).sort_values().plot(kind='barh', color='teal')
plt.title('Top 10 Features Driving Track Popularity')
plt.show()

## Step 6: Interpretation & Conclusion

### Key Findings:
1. **Artist Dominance:** The Feature Importance plot clearly shows that `artist_popularity` and `log_artist_followers` are the most significant predictors. This suggests that *who* releases the song matters more than the song's format (duration/explicit).
2. **Recency Effect:** `release_year` is often a top predictor, indicating that newer songs (or specific trending eras) are algorithmically favored.
3. **Explicit Content:** The statistical test (T-Test) helps validate whether explicit content actually correlates with popularity in this specific dataset sample.

### Future Improvements:
1. **NLP Analysis:** Perform sentiment analysis on `track_name` to see if certain keywords drive clicks.
2. **Audio Features:** Integrate API data for features like `danceability` and `energy` for a more content-based analysis.
3. **Hyperparameter Tuning:** Use GridSearch to optimize the Random Forest parameters.

### Conclusion:
This project successfully applied Regression Analysis to predict Spotify popularity. By implementing robust data cleaning (handling text booleans and genres) and feature engineering (log transforms), we established a reliable baseline model that explains the variance in song success.